# 04 – Combine niche model with SeaBee remote sensing

Combines the **niche model** (03b) with **SeaBee remote sensing** (shallow water)
into one dataset, which is the input to 05.

- **Niche model**: `nkm-utbredelsesmodell_norge_2026_25833.geo.parquet` – rule-based
  classification from light/salinity/substrate/intertidal.
- **SeaBee remote sensing**:
  `marint_naturkart_habitat_substrat_vegetasjon_regelc_polygoner_uten_rutenett_norge_2024-2025_25833.gpkg`
  – the national regelC habitat product: observed habitat classes
  (ALGAE, ROCK, SEDIMENT, ANGIO, DEEP, ANTHRO) in shallow water.

The rules for how remote sensing overrides the niche model live in
`mnk.rules.classify_seabee` (one rule set, derived from the data model
`nkm_klassifiseringsregler`). ANGIO/ALGAE/ROCK/SEDIMENT are authoritative for
both substrate and light (`nkm_lys`).

**Performance**: the niche model is national (~2.7 M polygons). Only polygons
that actually overlap SeaBee habitat are touched – the rest pass through
unchanged without expensive geometry operations.

In [ ]:
import gc
import io
from pathlib import Path

import numpy as np
import geopandas as gpd
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import shapely

import mnk
import mnk.rules
import mnk.attributes
import mnk.vectorize
import mnk.utils

CRS = "EPSG:25833"
REPO = Path.cwd()

NICHE_PARQUET = "gs://niva-geodata/MarintNaturKart/results/nkm-utbredelsesmodell_norge_2026_25833.geo.parquet"
SEABEE_GPKG = REPO / "marint_naturkart_habitat_substrat_vegetasjon_regelc_polygoner_uten_rutenett_norge_2024-2025_25833.gpkg"

TORRFALL_PARQUET = "gs://niva-geodata/MarintNaturKart/results/torrfall-filled_norge_25833.geo.parquet"

# Habitat SeaBee classes that can override the niche model.
# DEEP/ANTHRO do not override (see mnk.rules.SEABEE_RULES).
HABITAT_CLASSES = ["ALGAE", "ROCK", "SEDIMENT", "ANGIO"]


## 1. Load data

In [ ]:
niche = gpd.read_parquet(NICHE_PARQUET)
assert niche.crs.to_epsg() == 25833
print(f"Niche model: {len(niche):,} polygons")

seabee = gpd.read_file(
    SEABEE_GPKG, layer="polygons", columns=["class_1", "usikkerhet_klasse"]
).to_crs(CRS)
seabee = seabee.rename(columns={"class_1": "klasse"})
seabee["_usikkerhet"] = seabee["usikkerhet_klasse"].replace({"middels": "medium"})
seabee_hab = seabee.loc[
    seabee["klasse"].isin(HABITAT_CLASSES), ["klasse", "_usikkerhet", "geometry"]
].reset_index(drop=True)
print(f"SeaBee habitat: {len(seabee_hab):,} polygons")
print(seabee["klasse"].value_counts())
print(seabee_hab["_usikkerhet"].value_counts())

del seabee
gc.collect()

## 2. SeaBee habitat outside the niche model

The nisjemodell does not reach the whole shallow zone (it clips tørrfall against its
own polygons, and intertidal areas with missing salinity get no polygon).
`seabee_only` is the SeaBee habitat area left after subtracting the nisjemodell
polygons; it is classified in §5. Its salinitet is `brakk` if the SeaBee polygon
touches a nisjemodell polygon with brakk salinitet (`nkm_lm-sa`), else `salt`.

In [ ]:
seabee_only = mnk.vectorize.subtract_layer(seabee_hab, niche, workers=4, progress=True)
seabee_only = seabee_only.explode(index_parts=False).reset_index(drop=True)
seabee_only = seabee_only[~seabee_only.geometry.is_empty & seabee_only.geometry.notna()]

brakk = niche.geometry.values[
    (niche[mnk.rules.COL_SALINITET] == mnk.rules.NKM_SALINITET["brakk"]).to_numpy()
]
b_hit = shapely.STRtree(brakk).query(seabee_only.geometry.values, predicate="intersects")
is_brakk = np.zeros(len(seabee_only), dtype=bool)
is_brakk[np.unique(b_hit[0])] = True
seabee_only["_sal"] = np.where(is_brakk, "brakk", "salt")
print(f"SeaBee-only (no nisjemodell): {len(seabee_only):,} ({is_brakk.sum():,} brakk)")
del brakk, b_hit, is_brakk

## 3. Reclassification of SeaBee pieces

`update_seabee` is applied to every piece under SeaBee habitat (§4 and §5):

- `mnk.attributes.apply_seabee` lets the SeaBee class set the axes it overrides
  (`mnk.attributes.SEABEE_AXIS_OVERRIDE`: substrate and light for
  ALGAE/ROCK/SEDIMENT/ANGIO) together with their `datakilde` token and usikkerhet (the
  SeaBee `usikkerhet_klasse`, `middels` renamed to `medium`). Salinity and tørrfall
  keep the niche model values. It also sets `nkm_seabee_klasse`, `nkm_sa-fa`
  (ALGAE ⇒ `ABC`, ROCK ⇒ `0B`) and `metode_usikkerhet` (`NKM_satelitt_habitat`, plus
  `|NKM_nisjedata_kvalitet` where a niche-model axis usikkerhet remains).
- `mnk.attributes.classify` derives the NiN type with `classify_seabee` from the
  class and the axes; pieces of non-marine classes are dropped.

`nkm_prosess`, `metode_klassifisering` and the combined `nkm_usikkerhetsklasse` (the
worst axis, since salinity still decides MA vs MC) are derived for all rows by
`finalize` in §6.

In [ ]:
def update_seabee(pieces):
    """Let the SeaBee class set its axes and reclassify; drop non-marine pieces."""
    pieces = mnk.attributes.classify(mnk.attributes.apply_seabee(pieces))
    return pieces[pieces["nin_kode_fkid"].notna()]

## 4. Overlay: SeaBee habitat × niche model

`mnk.vectorize.overlay_update` splits the niche polygons that overlap SeaBee habitat
into the part inside the habitat, which goes through `update_seabee`, and the
remainder, which stays niche model. Only overlapping pairs go through geometry
operations; all other polygons pass through unchanged.

In [ ]:
combined = mnk.vectorize.overlay_update(niche, seabee_hab, update_seabee, workers=4, progress=True)
print(f"Niche ⊕ SeaBee: {len(combined):,} polygons")
print(combined["nkm_seabee_klasse"].value_counts())

del niche, seabee_hab
gc.collect()

## 5. Classify SeaBee habitat with no nisjemodell underneath

For a habitat class the rules need only **salinitet** (§2) and **tørrfall**: light and
substrate come from the SeaBee class (the eufotisk/løst start values below are
overridden by `update_seabee`). Each polygon starts as `utenfor` and the part inside
the kartverket tørrfall (`TORRFALL_PARQUET`) is set to `innenfor` with
`overlay_update`. Salinity usikkerhet is NA, as salinity is inferred from neighbours
rather than modelled.

There is **no NA tørrfall canvas** and **no deep-sea canvas**: areas covered by neither
the nisjemodell nor SeaBee stay an honest hole rather than a NA-filled or guessed
polygon.

In [ ]:
TF = mnk.rules.COL_TOERFALL
so = seabee_only.assign(**{
    mnk.rules.COL_LYS: mnk.rules.NKM_LYS["eufotisk"],
    mnk.rules.COL_SALINITET: seabee_only["_sal"].map(mnk.rules.NKM_SALINITET),
    mnk.rules.COL_SUBSTRAT: mnk.rules.NKM_SUBSTRAT["løst"],
    TF: mnk.rules.NKM_TOERFALL["utenfor"],
})
so["datakilde"] = mnk.attributes.build_datakilde(mnk.attributes.SOURCE_SUBSTRAT_NIVA, so[TF])
for col in mnk.attributes.USIKKERHET_AXIS_COLS:
    so[col] = pd.Series(pd.NA, index=so.index, dtype="string")

torrfall = gpd.read_parquet(TORRFALL_PARQUET).to_crs(CRS)[["geometry"]].reset_index(drop=True)
torrfall["geometry"] = shapely.make_valid(torrfall.geometry.to_numpy())
so = mnk.vectorize.overlay_update(
    so,
    torrfall,
    lambda p: p.assign(
        **{TF: mnk.rules.NKM_TOERFALL["innenfor"]},
        datakilde=mnk.attributes.set_kilde(p["datakilde"], mnk.attributes.SOURCE_TOERFALL_KARTVERKET),
    ),
)
del torrfall
gc.collect()

so = update_seabee(so)
innenfor = (so[TF] == mnk.rules.NKM_TOERFALL["innenfor"]).sum()
print(f"SeaBee-only pieces: {len(so):,} ({innenfor:,} innenfor tørrfall)")

## 6. Combine

`final = niche ⊕ SeaBee (§4) + SeaBee-only pieces (§5)`. `mnk.attributes.finalize`
derives `nkm_prosess`, `metode_klassifisering` (the `|`-joined k_metode codes of the
`datakilde` sources) and the combined `nkm_usikkerhetsklasse` for every row, and drops
the working columns (`_*`, the SeaBee `klasse`). `nkm_seabee_klasse` and `nkm_sa-fa`
are NA for the pure nisjemodell pieces.

In [ ]:
final = gpd.GeoDataFrame(pd.concat([combined, so], ignore_index=True), crs=CRS)
final = mnk.attributes.finalize(final)

print(f"Combined (niche ⊕ SeaBee): {len(final):,} polygons ({len(so):,} SeaBee-only)")
print(final["nin_kode_fkid"].value_counts())
print(final["metode_klassifisering"].value_counts())

del combined, seabee_only, so
gc.collect()

## 7. Save

In [ ]:
fname = mnk.utils.to_filename("nisjemodel-remote-sensing", "norge", "2026", CRS.split(":")[1])
parquet_path = REPO / f"{fname}.geo.parquet"

obj_cols = final.select_dtypes("object").columns
final[obj_cols] = final[obj_cols].astype("string")

_meta = io.BytesIO()
final.iloc[:0].to_parquet(_meta)
_meta.seek(0)
geo_meta = pq.read_schema(pa.PythonFile(_meta, mode="r")).metadata[b"geo"]
schema = pa.table(final.iloc[:0].to_arrow(geometry_encoding="WKB")).schema.with_metadata(
    {b"geo": geo_meta}
)

CHUNK = 100_000
with pq.ParquetWriter(parquet_path, schema, compression="snappy") as w:
    for start in range(0, len(final), CHUNK):
        part = pa.table(final.iloc[start : start + CHUNK].to_arrow(geometry_encoding="WKB"))
        w.write_table(part.cast(schema))
        del part
        gc.collect()
print(f"Saved {len(final):,} polygons -> {parquet_path.name}")